# Defter 8 — İklim projeksiyonları (CMIP6) ve son modeller
**Embedding tree growth theory into AI** · Prof. Dr. İlker Ercanlı

1. **Son modeller:** U+S, T+S ve yerel uyumlu T+S, 1901–2024 verisinin tamamıyla × 5 tohum; ayarlı XGBoost ve karma model. (Uygulama/GitHub için de bunlar kullanılacak.)
2. **CMIP6:** GFDL-ESM4, IPSL-CM6A-LR, MPI-ESM1-2-HR, MRI-ESM2-0, UKESM1-0-LL; tarihsel + SSP1-2.6, SSP2-4.5, SSP5-8.5; aylık tas, pr, hurs.
   Google Cloud'daki açık CMIP6 arşivinden okunur (lisans: CC BY 4.0). Fark yöntemi: 1985–2014'e göre değişim CRU iklimine eklenir.
3. **Ana sonuç (B planı):** sabit boyutta iklim etkisi — her ağaç bugünkü yarıçapında, 2041–2070 ve 2071–2100 iklimi ile 1985–2014 iklimi karşılaştırılır.
   İki uyum varsayımı: alan normalleri sabit (bugünkü uyum) / güncellenmiş (popülasyonlar yeni iklime uyum sağlar).
4. **Ekstrapolasyon işareti:** gelecek normallerinin eğitim alanlarına göre yeniliği (Mahalanobis > 1).
5. **İkincil sonuç:** ağaçları 2025–2100 yıl yıl büyütme (SSP2-4.5, SSP5-8.5); gözlenen boyut aralığının dışına çıkan ağaçlar işaretlenir.

Tablolar `ANALIZ_SON/H0…H5_*`. **Çalışma zamanı:** T4 ya da L4 GPU, "Yüksek RAM". Süre ≈ 3–4 saat (eğitim ≈ 2 saat, CMIP6 indirme ≈ 15–30 dk).
Kesilirse yeniden "Tümünü çalıştır": bitmiş eğitimler, indirilen iklim dosyaları ve tamamlanan senaryolar atlanır.

In [ ]:
# --- Kurulum ---
import os, json, time, math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:
    ANA = os.environ.get('ITRDB_AI', '.')
SONUC = os.path.join(ANA, 'SONUC_SON'); os.makedirs(SONUC, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Cihaz:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(GPU yok: Çalışma zamanı → T4 GPU seçin)')

AYAR = dict(batch=32768, max_epoch=25, sabir=3, lr=2e-3, lr_gizil=1e-2, J=8, gizli=128, emb=8,
            takson_dusur=0.10, su_onsel=0.5, tau_onsel=0.5, sv_onsel=0.5, desc_gurultu=0.2, wd=1e-4,
            rastgele_etki=False,   # False: popülasyon modeli (ana analiz); True: ağaç/alan rastgele etkili duyarlılık analizi
            r0_bas_cm=2.0, sifir_mm=0.005, hizli_deneme=False)

In [ ]:
# --- Veri ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

# Halkalar: 1901 halkaları çıkar (önceki yılın iklimi yok)
k = d['ring_year'] > YIL0
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k], torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
ser_zaman = t(d['series_zaman'], torch.bool)
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# Aylık iklim özellikleri [alan, ay(124*12), kanal]: T, log(1+P), M, VPD, gE
clim = d['clim']; soil = d['soil']; gE = d['gE']
NY = clim.shape[1]
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
OZ = t(ozel); del ozel, clim
oz_ort = OZ.reshape(-1, 5).mean(0); oz_ss = OZ.reshape(-1, 5).std(0)
desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
# Yetişme ortamı: 11 sürekli değişken (standartlaştırılmış) + biyom, ekolojik bölge, Köppen sınıfları (son indeks = bilinmeyen)
env = d['env']; ENV = t((env - env.mean(0)) / env.std(0), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
KAT_GOR = {}   # bölünmeye göre eğitimde görülen sınıflar (görülmeyenler 'bilinmeyen' gömmesine gider)
ORTAM_BOYUT = ENV.shape[1] + 3 * 4

class OrtamGirdi(nn.Module):
    """Alan başına yetişme ortamı vektörü: sürekli değişkenler + biyom/ekolojik bölge/Köppen gömmeleri (her biri 4 boyut)"""
    def __init__(self):
        super().__init__()
        self.b = nn.Embedding(NB + 1, 4); self.e = nn.Embedding(NE + 1, 4); self.k = nn.Embedding(NK + 1, 4)
    def forward(self, G):
        b, e, k = G['b'], G['e'], G['k']
        if KAT_GOR:
            b = torch.where(KAT_GOR['b'][b], b, torch.full_like(b, NB)); e = torch.where(KAT_GOR['e'][e], e, torch.full_like(e, NE))
            k = torch.where(KAT_GOR['k'][k], k, torch.full_like(k, NK))
        if self.training:      # %10 rastgele maskeleme: 'bilinmeyen' gömmesi de öğrenilsin
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        x = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if self.training else 0)
        return torch.cat([x, self.b(b), self.e(e), self.k(k)], 1)
print(f'{NR:,} halka, {NSER:,} seri, {NTREE:,} ağaç, {S} alan — veri {"GPU" if DEV=="cuda" else "CPU"} belleğinde')

# Yerel uyum için alan nem normali: 1901–1960 aylık toprak nemi ortalaması (zamansal testte de yalnız eğitim dönemi)
MBAR = OZ[:, :60 * 12, 2].mean(1); MREF = MBAR.mean()
def alan_girdi(s):
    """Alan düzeyi girdiler (SHAP ve ihlal analizinde gruplar halinde değiştirilebilsin diye sözlük)"""
    return dict(g=site_genus[s], sp=site_species[s], desc=DESC[s], env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s], mbar=MBAR[s])

def girdiler(idx):
    """Halka düzeyi girdiler: 24 aylık iklim penceresi X, birikmiş yarıçap rp, seri ve alan girdileri"""
    X, s = pencere(idx); G = alan_girdi(s); G.update(X=X, rp=R_rp[idx], ser=R_ser[idx]); return G

def pencere(idx):
    """Halka idx için 24 aylık pencere (önceki yıl Ocak – bu yıl Aralık): [B, 24, 5]"""
    s = ser_site[R_ser[idx]]
    bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

In [ ]:
# --- Test bölünmeleri ---
site_bolge = np.array(soz['site_bolge']); site_kat = d['site_blok_kat']; site_takson = d['site_takson_test']
site_ic = d['site_ic_dogrulama']

def bolunme(ad):
    """Dönüş: (eğitim, doğrulama, test) halka indeksleri ve test alanlarında takson gizlensin mi"""
    site_r = ser_site[R_ser].cpu().numpy(); yil = R_yil.cpu().numpy(); zuy = ser_zaman[R_ser].cpu().numpy()
    gizle = False
    if ad == 'zaman':
        test = zuy & (yil > 1960); havuz = yil <= 1960
    elif ad.startswith('bolge:'):
        tb = np.isin(site_r, np.where(site_bolge == ad.split(':')[1])[0]); test = tb; havuz = ~tb
    elif ad.startswith('blok:'):
        tb = np.isin(site_r, np.where(site_kat == int(ad.split(':')[1]))[0]); test = tb; havuz = ~tb
    elif ad == 'takson':
        tb = np.isin(site_r, np.where(site_takson)[0]); test = tb; havuz = ~tb; gizle = True
    elif ad.startswith('iklim:'):
        # İklim dışa-genelleme: en sıcak (1901–1960 ort. sıcaklık) ya da en kurak (P/PET) %10 alan yalnız testte
        tur = ad.split(':')[1]
        if tur == 'kurak2':          # %10–20 kurak bant test; eğitim yalnız en kurak %20 dışı alanlarla
            x = -desc[:, 5]; q8, q9 = np.quantile(x, [0.8, 0.9])
            test = np.isin(site_r, np.where((x >= q8) & (x < q9))[0]); havuz = np.isin(site_r, np.where(x < q8)[0])
        else:
            x = {'sicak': desc[:, 2], 'kurak': -desc[:, 5], 'nemli': desc[:, 5]}[tur]
            tb = np.isin(site_r, np.where(x >= np.quantile(x, 0.9))[0]); test = tb; havuz = ~tb
    elif ad == 'tum':            # projeksiyon için son modeller: tüm alanlar ve yıllar; test = iç doğrulama alanları
        havuz = np.ones(len(site_r), bool); test = site_ic[site_r].copy()
    else:
        raise ValueError(ad)
    ic = site_ic[site_r]
    egit = np.where(havuz & ~ic)[0]; dog = np.where(havuz & ic)[0]; test = np.where(test)[0]
    return t(egit, torch.long), t(dog, torch.long), t(test, torch.long), gizle

In [ ]:
# --- Modeller ---
class AlanAgi(nn.Module):
    """z_s -> parametreler. Cins/tür gömmeleri + standartlaştırılmış alan tanımlayıcıları."""
    def __init__(self, n_cikis, emb, gizli, sutun=None, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.sutun = list(range(DESC.shape[1])) if sutun is None else sutun   # saf teori: yalnız [enlem, yükselti]
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)   # son indeks = "bilinmeyen takson"
        self.mlp = nn.Sequential(nn.Linear(2 * emb + len(self.sutun) + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, n_cikis))
    def forward(self, G, gizle_maske=None):
        g, sp = G['g'], G['sp']
        if gizle_maske is not None:
            g = torch.where(gizle_maske, torch.full_like(g, NG), g); sp = torch.where(gizle_maske, torch.full_like(sp, NS), sp)
        d = G['desc'][:, self.sutun]
        d = d + (AYAR['desc_gurultu'] * torch.randn_like(d) if self.training else 0)
        parcalar = [self.ge(g), self.se(sp), d] + ([self.ortam(G)] if self.ortam is not None else [])
        return self.mlp(torch.cat(parcalar, 1))

def monoton(x, a, c, s):
    """Denklem (4): negatif olmayan lojistik karışımı, 0–1 arası, azalmayan. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

def alan_gor_bul(seri_gor):
    m = torch.zeros(S, dtype=torch.bool, device=DEV); m[ser_site[seri_gor]] = True
    return m

class Gizil(nn.Module):
    """Seri başına gizil başlangıç yarıçapı r0 = softplus(ρ) ve ağaç etkisi u; hiyerarşik önsel cezası"""
    def __init__(self, r0_bas):
        super().__init__()
        rho0 = math.log(math.expm1(r0_bas))
        self.rho = nn.Parameter(torch.full((NSER,), rho0, device=DEV))
        self.mu_g = nn.Parameter(torch.full((NG + 1,), rho0, device=DEV))
        self.u = nn.Parameter(torch.zeros(NTREE, device=DEV))          # ağaç etkisi
        self.v = nn.Parameter(torch.zeros(S, device=DEV))              # alan etkisi (toprak, sıklık vb. kaydedilmemiş farklar)
        # Önsel ölçekleri sabit (zayıf bilgili) hiperparametreler: MAP ile öğrenilen varyans bileşenleri sıfıra çöker.
        # Yeni ağaç/seri için öngörüde kullanılan yayılım ise eğitilmiş ağaçların GÖRGÜL dağılımından alınır.
        self.su_onsel, self.tau_onsel = AYAR['su_onsel'], AYAR['tau_onsel']
        self.register_buffer('su_gorgul', torch.tensor(AYAR['su_onsel'], device=DEV))
        self.register_buffer('tau_gorgul', torch.tensor(AYAR['tau_onsel'], device=DEV))
        self.sv_onsel = AYAR['sv_onsel']
        self.register_buffer('sv_gorgul', torch.tensor(AYAR['sv_onsel'], device=DEV))
        self.register_buffer('ort_u', torch.tensor(0., device=DEV)); self.register_buffer('ort_v', torch.tensor(0., device=DEV))
        self.register_buffer('ort_d', torch.tensor(0., device=DEV))
    def su(self): return self.su_gorgul
    def tau(self): return self.tau_gorgul
    def gorgul_guncelle(self, seri_gor, agac_gor):
        with torch.no_grad():
            g = site_genus[ser_site]
            # Görgül dağılım: ortalama da alınır (ağın sabit terimi ile rastgele etkilerin ortalaması birbirine kayabilir)
            u, d, v = self.u[agac_gor], (self.rho - self.mu_g[g])[seri_gor], self.v[alan_gor_bul(seri_gor)]
            self.su_gorgul.copy_(u.std().clamp_min(0.02)); self.ort_u.copy_(u.mean())
            self.tau_gorgul.copy_(d.std().clamp_min(0.02)); self.ort_d.copy_(d.mean())
            self.sv_gorgul.copy_(v.std().clamp_min(0.02)); self.ort_v.copy_(v.mean())
    def ceza(self, seri_gor, agac_gor):
        """Hiyerarşik önsel; YALNIZ eğitimde verisi olan seri ve ağaçlar üzerinden"""
        g = site_genus[ser_site]
        d = (self.rho - self.mu_g[g])[seri_gor]; u = self.u[agac_gor]; v = self.v[alan_gor_bul(seri_gor)]
        return ((d ** 2).sum() / (2 * self.tau_onsel ** 2) + (u ** 2).sum() / (2 * self.su_onsel ** 2)
                + (v ** 2).sum() / (2 * self.sv_onsel ** 2))
    def gizil_deger(self, ser, ornek=None):
        """Halka başına ρ ve u. ornek=(seri_gor, agac_gor, gizle): görülmemişler için önselden rastgele çekim"""
        rho, u, v = self.rho[ser], self.u[ser_tree[ser]], self.v[ser_site[ser]]
        if ornek is not None:
            seri_gor, agac_gor, gizle, mu_glob = ornek
            g = site_genus[ser_site[ser]]
            mu = torch.where(gizle, mu_glob, self.mu_g[g]) if gizle is not None else self.mu_g[g]
            rho = torch.where(seri_gor[ser], rho, mu + self.ort_d + self.tau() * torch.randn_like(rho))
            u = torch.where(agac_gor[ser_tree[ser]], u, self.ort_u + self.su() * torch.randn_like(u))
            v = torch.where(alan_gor_bul(seri_gor)[ser_site[ser]], v, self.ort_v + self.sv_gorgul * torch.randn_like(v))
        return rho, u + v

class TeoriModeli(nn.Module):
    def __init__(self, J=8, emb=8, gizli=128, ablasyon=None, r0_bas=2.0, m_bas=2/3, vpd=False, m_sabit=None, saf=False, ortam=False, uyum=False, genis=False, tur=False):
        super().__init__()
        self.J, self.abl = J, ablasyon
        self.uyum = uyum
        self.mgen = 0.3 if genis else 0.1   # nem eşiği merkezlerinin kayma aralığı
        self.m_logit = math.log(m_bas / (1 - m_bas)); self.vpd = vpd; self.m_sabit = m_sabit
        n = 3 + 6 * J + 24 + 1 + 1          # +1: başlangıç yarıçapı r0 (alan ve takson düzeyinde, ağdan)
        if ablasyon == 'abl_min': n += 3
        if vpd: n += 3 * J                 # VPD yanıtı (üçüncü sınırlayıcı faktör)
        if uyum: n += 1                    # yerel uyum katsayısı λ
        self.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        self.ag = AlanAgi(n, emb, gizli, sutun=[] if tur else ([0, 1] if saf else None), ortam=ortam)   # saf: iklim normalleri parametre ağına girmez
        if ablasyon == 'abl_mono':      # monoton yanıt yerine kısıtsız küçük ağ (girdi: x ve alan parametreleri)
            self.serbT = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
            self.serbM = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
        self.gz = Gizil(r0_bas)
        self.Tbaz = torch.linspace(-5, 25, J, device=DEV); self.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)

    def parametreler(self, G, gizle=None):
        if torch.is_tensor(G): G = alan_girdi(G)
        o = self.ag(G, gizle); J = self.J
        p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm² (başlangıç: r ≈ 56 cm)
        p['m'] = torch.sigmoid(o[:, 2] + self.m_logit)                            # başlangıç varsayılan 2/3
        if self.m_sabit is not None: p['m'] = torch.full_like(p['m'], self.m_sabit)   # profil olabilirlik: m sabit
        if getattr(self, 'm_cins', None) is not None:   # sentetik veri üretimi: m cinse göre dışarıdan; η aynı ortalama büyümeyi korur
            m_yeni = self.m_cins[G['g']]; p['eta'] = p['eta'] * 300.0 ** (p['m'] - m_yeni); p['m'] = m_yeni
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], self.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], self.Mbaz + self.mgen * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if self.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], self.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['log_sigma'] = o[:, i] - 0.3; i += 1
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1   # cm
        if self.abl == 'abl_min': p['karisim'] = torch.softmax(o[:, i:i+3], 1)
        if self.uyum: p['lam'] = torch.sigmoid(o[:, -1])          # 0 = mutlak tepki, 1 = tamamen yerel uyumlu
        return p

    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = self.parametreler(G, gizle); G0 = G
        ser = G['ser']
        rho_i, u_i = self.gz.gizil_deger(ser, ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        r0 = 0.05 if self.abl == 'abl_r0' else (F.softplus(rho_i) if AYAR['rastgele_etki'] else p['r0'])
        r = r0 + G['rp']; B = math.pi * r ** 2                                   # cm²
        # Denklem (2): büyüme potansiyeli
        if self.abl == 'abl_G':
            G = p['eta'] * 30.0
        else:
            x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
            G = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20
        # Denklem (3)–(4): sınırlayıcı faktör
        T, M, gEk = X[..., 0], X[..., 2], X[..., 4]
        if self.uyum: M = M - p['lam'][:, None] * (G0['mbar'][:, None] - MREF)   # eşik alanın nem normaline göre kayar
        if self.abl == 'abl_mono':
            pT = torch.cat([p['aT'], p['cT'], p['sT']], 1); pM = torch.cat([p['aM'], p['cM'], p['sM']], 1)
            gT = torch.sigmoid(self.serbT(torch.cat([T[..., None] / 10, pT[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
            gM = torch.sigmoid(self.serbM(torch.cat([M[..., None] * 5, pM[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
        else:
            gT = monoton(T, p['aT'], p['cT'], p['sT']); gM = monoton(M, p['aM'], p['cM'], p['sM'])
        if self.abl == 'abl_min':
            k = p['karisim'][:, None, :]
            sinir = k[..., 0] * gT + k[..., 1] * gM + k[..., 2] * gT * gM
        else:
            sinir = torch.minimum(gT, gM)
        if self.vpd:     # Liebig: min(g_T, g_M, g_D); g_D = 1 − monoton(VPD) → atmosfer kurudukça büyüme azalır
            gD = 1 - monoton(X[..., 3], p['aD'], p['cD'], p['sD'])
            sinir = torch.minimum(sinir, gD)
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        BAI = G * C * torch.exp(u_i)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                         # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        return mu, p['log_sigma'], dict(gT=gT, gM=gM, C=C, G=G, lam=p.get('lam'))

class SerbestAg(nn.Module):
    """Kısıtsız karşılaştırma: 24 aylık iklim -> zamansal evrişim; + log yarıçap + alan gömmesi -> log w"""
    def __init__(self, emb=8, gizli=40, r0_bas=2.0, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.conv = nn.Sequential(nn.Conv1d(5, 32, 3), nn.GELU(), nn.Conv1d(32, 32, 3), nn.GELU())
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)
        self.mlp = nn.Sequential(nn.Linear(32 * 20 + 2 * emb + DESC.shape[1] + 1 + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, 2))
        self.gz = Gizil(r0_bas)
    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = (G['X'] - oz_ort) / oz_ss
        h = self.conv(X.transpose(1, 2)).flatten(1)
        g, sp = G['g'], G['sp']
        if gizle is not None:
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        lr = torch.log1p(G['rp'])[:, None]
        d = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if self.training else 0)
        parcalar = [h, self.ge(g), self.se(sp), d, lr] + ([self.ortam(G)] if self.ortam is not None else [])
        o = self.mlp(torch.cat(parcalar, 1))
        _, u_i = self.gz.gizil_deger(G['ser'], ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        return o[:, 0] - 0.3 + u_i, o[:, 1] - 0.3, {}

def yap(ad):
    """Model adı: taban ad + isteğe bağlı '_ortam' eki (yetişme ortamı girdileri)"""
    ortam = ad.endswith('_ortam'); ad = ad[:-6] if ortam else ad
    T = lambda **k: TeoriModeli(AYAR['J'], AYAR['emb'], AYAR['gizli'], k.pop('abl', None), AYAR['r0_bas_cm'], ortam=ortam, **k).to(DEV)
    if ad == 'serbest': return SerbestAg(AYAR['emb'], r0_bas=AYAR['r0_bas_cm'], ortam=ortam).to(DEV)
    if ad == 'teori': return T()
    if ad == 'teori_vpd': return T(vpd=True)
    if ad == 'teori_uyum': return T(uyum=True)
    if ad == 'teori_genis': return T(genis=True)
    if ad == 'teori_tur': return T(tur=True)
    if ad == 'teori_saf': return T(vpd=True, saf=True)
    if ad.startswith('teori_mf'):       # profil olabilirlik: teori_mf050 -> m = 0.50 sabit
        return T(m_sabit=int(ad[8:]) / 1000 if len(ad) == 12 else int(ad[8:]) / 100)
    if ad.startswith('teori_m'): return T(m_bas=int(ad[7:]) / 100)
    if ad.startswith('abl_'): return T(abl=ad)
    raise ValueError(ad)

def ag_param_sayisi(m):
    return sum(p.numel() for n, p in m.named_parameters() if not n.startswith('gz.'))

for ad in ['teori_ortam', 'teori_tur']:
    print(f'{ad:16s} eğitilebilir ağ parametresi: {ag_param_sayisi(yap(ad)):,}')

In [ ]:
# --- Eğitim ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm'])
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

LOG_ESIK = math.log(0.01)            # ölçüm çözünürlüğü: sıfır halka = "0,01 mm'den küçük" (sansürlü gözlem)
def nll(mu, ls, y):
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, seri_gor, agac_gor, gizle_alan=False, K=8, bs=131072):
    """Görülmüş ağaçlar: koşullu öngörü. Görülmemiş ağaçlar (yeni alan/tür): ρ ve u önselden K kez çekilir;
    öngörü = karışımın ortalaması, varyans = ort(σ²) + var(μ)  (log ölçeğinde)"""
    model.eval(); MU, SG = [], []
    g_gor = torch.unique(site_genus[ser_site[seri_gor.nonzero().squeeze(1)]])
    mu_glob = model.gz.mu_g[g_gor].mean()
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            b = idx[i:i+bs]
            gz = torch.ones(len(b), dtype=torch.bool, device=DEV) if gizle_alan else None
            yeni = ~agac_gor[ser_tree[R_ser[b]]] | ~seri_gor[R_ser[b]]
            if not AYAR['rastgele_etki']: yeni = torch.zeros_like(yeni)
            if not yeni.any():
                mu, ls, _ = model(b, gz); MU.append(mu); SG.append(torch.exp(ls)); continue
            ms, vs = [], []
            for k in range(K):
                mu, ls, _ = model(b, gz, ornek=(seri_gor, agac_gor, gz, mu_glob)); ms.append(mu); vs.append(torch.exp(2 * ls))
            ms, vs = torch.stack(ms), torch.stack(vs)
            MU.append(ms.mean(0)); SG.append(torch.sqrt(vs.mean(0) + ms.var(0, unbiased=False)))
    model.train()
    return torch.cat(MU), torch.cat(SG)

def kat_gor_ayarla(egt):
    """Eğitim halkalarının alanlarında görülen biyom / ekolojik bölge / Köppen sınıfları"""
    sg = torch.unique(ser_site[R_ser[egt]]); KAT_GOR.clear()
    for anahtar, dizi, n in [('b', site_biyom, NB), ('e', site_eko, NE), ('k', site_kg, NK)]:
        m = torch.zeros(n + 1, dtype=torch.bool, device=DEV); m[dizi[sg]] = True; KAT_GOR[anahtar] = m

def egit(bolunme_ad, model_ad, tohum):
    is_ad = f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}'
    klasor = os.path.join(SONUC, is_ad); os.makedirs(klasor, exist_ok=True)
    if os.path.exists(os.path.join(klasor, 'bitti.json')):
        print('atlandı (bitmiş):', is_ad); return
    torch.manual_seed(tohum); np.random.seed(tohum)
    egt, dog, test, gizle = bolunme(bolunme_ad)
    kat_gor_ayarla(egt)
    if AYAR['hizli_deneme']:
        egt, dog, test = egt[torch.randperm(len(egt), device=DEV)[:200000]], dog[:20000], test[:20000]
    model = yap(model_ad)
    gizil = [p for n, p in model.named_parameters() if n.startswith('gz.')]
    ag = [p for n, p in model.named_parameters() if not n.startswith('gz.')]
    opt = torch.optim.AdamW([{'params': ag, 'lr': AYAR['lr'], 'weight_decay': AYAR['wd']},
                             {'params': gizil, 'lr': AYAR['lr_gizil'] if AYAR['rastgele_etki'] else 0.0, 'weight_decay': 0.0}])
    kayit = os.path.join(klasor, 'son.pt'); bas_ep, en_iyi, sabir, gecmis = 0, float('inf'), 0, []
    if os.path.exists(kayit):
        ck = torch.load(kayit, map_location=DEV)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        bas_ep, en_iyi, sabir, gecmis = ck['epoch'] + 1, ck['en_iyi'], ck['sabir'], ck['gecmis']
        print(f'{is_ad}: {bas_ep}. epoch\'tan devam')
    n_egt = len(egt)
    agac_gor = torch.zeros(NTREE, dtype=torch.bool, device=DEV); agac_gor[ser_tree[R_ser[egt]]] = True
    seri_gor = torch.zeros(NSER, dtype=torch.bool, device=DEV); seri_gor[R_ser[egt]] = True
    for ep in range(bas_ep, AYAR['max_epoch']):
        t0 = time.time(); kar = egt[torch.randperm(n_egt, device=DEV)]; top = 0.
        for i in range(0, n_egt, AYAR['batch']):
            b = kar[i:i+AYAR['batch']]
            gz = (torch.rand(len(b), device=DEV) < AYAR['takson_dusur'])
            mu, ls, _ = model(b, gz)
            kayip_v = nll(mu, ls, hedef(b))
            kayip = kayip_v + (model.gz.ceza(seri_gor, agac_gor) / n_egt if AYAR['rastgele_etki'] else 0.)
            opt.zero_grad(set_to_none=True); kayip.backward()
            torch.nn.utils.clip_grad_norm_(ag, 5.0); opt.step(); top += kayip_v.item() * len(b)
        model.gz.gorgul_guncelle(seri_gor, agac_gor)
        mu, sg = tahmin(model, dog, seri_gor, agac_gor)
        yd = hedef(dog); v = nll(mu, torch.log(sg), yd).item()
        nz = yd > LOG_Y_SIFIR + 1e-6; v_rmse = torch.sqrt(((mu - yd)[nz] ** 2).mean()).item(); v_sg = torch.sqrt((sg ** 2).mean()).item()
        gecmis.append(dict(epoch=ep, egitim=top / n_egt, dogrulama=v, sn=time.time() - t0))
        ek = (f'  σ_alan {model.gz.sv_gorgul.item():.2f}  σ_ağaç {model.gz.su().item():.2f}' if AYAR['rastgele_etki'] else '')
        print(f'{is_ad} ep {ep:2d}  eğitim {top/n_egt:.4f}  doğrulama {v:.4f} (RMSE {v_rmse:.3f}, öngörü σ {v_sg:.3f}){ek}  ({time.time()-t0:.0f} sn)')
        if v < en_iyi - 1e-4:
            en_iyi, sabir = v, 0; torch.save(model.state_dict(), os.path.join(klasor, 'en_iyi.pt'))
        else:
            sabir += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep, en_iyi=en_iyi, sabir=sabir, gecmis=gecmis), kayit)
        if sabir >= AYAR['sabir']: break
    # Test tahminleri (en iyi model)
    model.load_state_dict(torch.load(os.path.join(klasor, 'en_iyi.pt'), map_location=DEV))
    mu, sg = tahmin(model, test, seri_gor, agac_gor, gizle_alan=gizle)
    y = hedef(test)
    np.savez_compressed(os.path.join(klasor, 'test_tahmin.npz'), idx=test.cpu().numpy().astype(np.int32),
                        mu=mu.cpu().numpy().astype(np.float16), sigma=sg.cpu().numpy().astype(np.float16))
    nz = y > LOG_Y_SIFIR + 1e-6         # nokta ölçütleri sıfır olmayan halkalarda; NLL sansürlü olanları da içerir
    ozet = dict(is_ad=is_ad, n_egitim=n_egt, n_test=len(test), rmse_log=float(torch.sqrt(((mu - y)[nz] ** 2).mean())),
                bias_log=float((mu - y)[nz].mean()), nll=float(nll(mu, torch.log(sg), y)),
                kapsama90=float(((y - mu).abs() < 1.645 * sg)[nz].float().mean()), en_iyi_dogrulama=en_iyi,
                ag_param=ag_param_sayisi(model), gecmis=gecmis, ayar=AYAR)
    if not model_ad.startswith('serbest'):           # alan parametreleri (m, eşikler, ω) — şekiller için
        with torch.no_grad():
            p = model.parametreler(torch.arange(S, device=DEV), torch.full((S,), gizle, device=DEV) & t(site_takson, torch.bool))
        np.savez_compressed(os.path.join(klasor, 'alan_param.npz'), **{k: v.cpu().numpy() for k, v in p.items()},
                            r0_cm=F.softplus(model.gz.rho).detach().cpu().numpy(), u=model.gz.u.detach().cpu().numpy())
    json.dump(ozet, open(os.path.join(klasor, 'bitti.json'), 'w'), indent=1)
    os.remove(kayit)
    print(f'BİTTİ {is_ad}: test RMSE(log) {ozet["rmse_log"]:.3f}, kapsama %90 → {ozet["kapsama90"]:.3f}')

In [ ]:
# --- Ortak araçlar: topluluk tahmini, ölçütler, bootstrap ---
import glob, pandas as pd
from scipy.stats import norm
ANALIZ = os.path.join(ANA, 'ANALIZ_SON'); os.makedirs(ANALIZ, exist_ok=True)
Yn = hedef(torch.arange(NR, device=DEV)).cpu().numpy(); NZn = (R_w > 0).cpu().numpy()
SITE_n = ser_site[R_ser].cpu().numpy(); YIL_n = R_yil.cpu().numpy()
def topluluk(b, m):
    MU, V, idx = [], [], None
    for kl in sorted(glob.glob(os.path.join(SONUC, f'{b}__{m}__s*'))):
        if not os.path.exists(os.path.join(kl, 'bitti.json')): continue
        z = np.load(os.path.join(kl, 'test_tahmin.npz'))
        if idx is None: idx = z['idx']
        MU.append(z['mu'].astype(np.float32)); V.append(z['sigma'].astype(np.float32) ** 2)
    if not MU: return None
    MU, V = np.stack(MU), np.stack(V)
    return idx, MU.mean(0), np.sqrt(V.mean(0) + MU.var(0)), len(MU)
def olcutler(y, mu, sg, nz):
    r = (y - mu)[nz]; z = (y - mu) / sg
    nllv = np.where(nz, 0.5 * z**2 + np.log(sg) + 0.9189385, -norm.logcdf((math.log(0.01) - mu) / sg))
    crps = sg * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / math.sqrt(math.pi))
    return dict(RMSE=float(np.sqrt((r**2).mean())), bias=float(r.mean()), NLL=float(nllv.mean()),
                CRPS=float(crps[nz].mean()), kapsama90=float((np.abs(z) < 1.645)[nz].mean()))
rng = np.random.default_rng(2026)
def boot_fark(y, ma, mb, nz, grup, B=1000):
    """RMSE(a) − RMSE(b) ve grup (alan ya da yıl) bootstrap aralıkları"""
    g = grup[nz]; ea, eb = (y - ma)[nz] ** 2, (y - mb)[nz] ** 2
    u, inv = np.unique(g, return_inverse=True)
    Sa, Sb, n = np.bincount(inv, ea, len(u)), np.bincount(inv, eb, len(u)), np.bincount(inv, None, len(u))
    W = rng.multinomial(len(u), np.full(len(u), 1 / len(u)), size=B).astype(float)
    d = np.sqrt(W @ Sa / (W @ n)) - np.sqrt(W @ Sb / (W @ n))
    return float(np.sqrt(Sa.sum() / n.sum()) - np.sqrt(Sb.sum() / n.sum())), np.quantile(d, [0.025, 0.975]), np.quantile(d, [0.05, 0.95])
def alan_agirlikli_rmse(y, mu, nz, s):
    u, inv = np.unique(s[nz], return_inverse=True)
    return float(np.sqrt(np.bincount(inv, (y - mu)[nz] ** 2) / np.bincount(inv)).mean())

In [ ]:
# --- Senaryo ve özellik altyapısı (Defter 5–7 ile aynı) ---
from scipy.stats import spearmanr
import gc
CL = d['clim']; LAT = d['site_lat']
def gun_uzunlugu(lat):
    gun = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
    dek = 23.45 * np.sin(np.deg2rad(360 * (284 + gun) / 365))
    x = -np.tan(np.deg2rad(lat))[:, None] * np.tan(np.deg2rad(dek))[None, :]
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))
def isi_indeksi(T):
    S_ = T.shape[0]; Tclim = np.nanmean(T.reshape(S_, -1, 12), axis=1)
    return np.sum(np.where(Tclim > 0, (np.maximum(Tclim, 0) / 5) ** 1.514, 0), axis=1)
def kova(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    """Defter 1 ile aynı VS-Lite kova modeli; ısı indeksi I dışarıdan (taban iklim) verilir"""
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for t in range(N):
        Tt, Pt, Lt = T[:, t], P[:, t], L[:, t % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, t] = m
    return M
es = lambda T: 0.6108 * np.exp(17.27 * T / (T + 237.3))
@torch.no_grad()
@torch.no_grad()
def tahmin_G(model, G, bs=65536):
    n = len(G['rp']); out = []
    for i in range(0, n, bs): out.append(model(None, None, G={k: v[i:i+bs] for k, v in G.items()})[0])
    return torch.cat(out)
def yukle(bolunme_ad, model_ad, tohum=0):
    kl = os.path.join(SONUC, f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}')
    if not os.path.exists(os.path.join(kl, 'en_iyi.pt')): return None, None
    egt, _, test, _ = bolunme(bolunme_ad); kat_gor_ayarla(egt)
    m = yap(model_ad); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval()
    return m, test

import xgboost as xgb
print('xgboost', xgb.__version__)
NG_SEC = 2_000_000   # XGBoost eğitim alt örneklemi (halka); doğrulama 300 bin
def ozellik_ag(b, gizle):
    """[B, 126]: 24 ay × (T, logP, M, VPD) + log yarıçap + g_E ağırlıklı mevsim ortalamaları (önceki/bu yıl) + alan + cins, tür"""
    X, s = pencere(b); w = X[..., 4]
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    g = site_genus[s].float(); sp = site_species[s].float()
    if gizle: g = torch.full_like(g, float('nan')); sp = torch.full_like(sp, float('nan'))
    return torch.cat([X[..., :4].flatten(1), torch.log(R_rp[b] + 0.1)[:, None]] + mev + [DESC[s], ENV[s], g[:, None], sp[:, None]], 1)
def ozellik_np(idx, gizle, bs=500_000):
    return np.concatenate([ozellik_ag(idx[i:i+bs], gizle).cpu().numpy() for i in range(0, len(idx), bs)]).astype(np.float32)
FT = ['q'] * (96 + 1 + 8 + DESC.shape[1] + ENV.shape[1]) + ['c', 'c']
XDEV = 'cuda' if DEV == 'cuda' else 'cpu'
def kaydet(b, ad, idx, mu, sg, ek):
    kl = os.path.join(SONUC, f'{b.replace(":", "-")}__{ad}__s0'); os.makedirs(kl, exist_ok=True)
    np.savez_compressed(os.path.join(kl, 'test_tahmin.npz'), idx=idx.astype(np.int32), mu=mu.astype(np.float16), sigma=sg.astype(np.float16))
    json.dump(ek, open(os.path.join(kl, 'bitti.json'), 'w'))
def bitti(b, ad): return os.path.exists(os.path.join(SONUC, f'{b.replace(":", "-")}__{ad}__s0', 'bitti.json'))

def dkm_ozellik(b, gizle, g_gor):
    """Doğrusal karma model tasarımı: sabit, log r, (log r)², 8 mevsim iklimi, DESC, ENV, cins (gizlide görülen cinslerin ortalaması)"""
    X, s = pencere(b); w = X[..., 4]
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    lr = torch.log(R_rp[b] + 0.1)[:, None]
    oh = F.one_hot(site_genus[s], NG + 1).float()
    if gizle: oh = g_gor.float()[None, :].expand(len(b), -1) / g_gor.sum()
    return torch.cat([torch.ones_like(lr), lr, lr ** 2] + mev + [DESC[s], ENV[s], oh], 1), s

In [ ]:
# --- Klasik modeller: yeniden kullanılabilir kurulum ve tahmin fonksiyonları ---
XGB_VARSAYILAN = dict(max_depth=10, eta=0.08, subsample=0.8, colsample_bytree=0.6, max_bin=128, min_child_weight=50)
def xgb_kur(b, prm=XGB_VARSAYILAN, n_egt=NG_SEC, n_tur=2000, nicel=True, tohum=0):
    egt, dog, test, gizle = bolunme(b)
    egt = egt[R_w[egt] > 0]; dog = dog[R_w[dog] > 0]
    g = torch.Generator(device=DEV).manual_seed(tohum)
    egt = egt[torch.randperm(len(egt), device=DEV, generator=g)[:n_egt]]; dog = dog[torch.randperm(len(dog), device=DEV, generator=g)[:300_000]]
    if AYAR['hizli_deneme']: egt, dog, test, n_tur = egt[:100_000], dog[:20_000], test[:20_000], 40
    dtr = xgb.DMatrix(ozellik_np(egt, False), hedef(egt).cpu().numpy(), feature_types=FT, enable_categorical=True)
    dva = xgb.DMatrix(ozellik_np(dog, False), hedef(dog).cpu().numpy(), feature_types=FT, enable_categorical=True)
    ort = dict(device=XDEV, tree_method='hist', max_cat_to_onehot=1, seed=tohum, **prm)
    m1 = xgb.train(dict(ort, objective='reg:squarederror'), dtr, n_tur, evals=[(dva, 'dog')], early_stopping_rounds=50, verbose_eval=False)
    m2 = None
    if nicel:
        m2 = xgb.train(dict(ort, objective='reg:quantileerror', quantile_alpha=np.array([0.05, 0.95])), dtr, min(n_tur, 600),
                       evals=[(dva, 'dog')], early_stopping_rounds=50, verbose_eval=False)
    v = float(np.sqrt(np.mean((m1.predict(dva, iteration_range=(0, m1.best_iteration + 1)) - dva.get_label()) ** 2)))
    del dtr, dva
    return dict(m1=m1, m2=m2, test=test, gizle=gizle, dog_rmse=v)
import gc
def xgb_tahmin(K, idx, gizle=False):
    MU, SG = [], []
    for i in range(0, len(idx), 1_000_000):
        dte = xgb.DMatrix(ozellik_np(idx[i:i+1_000_000], gizle), feature_types=FT, enable_categorical=True)
        MU.append(K['m1'].predict(dte, iteration_range=(0, K['m1'].best_iteration + 1)))
        if K['m2'] is not None:
            q = K['m2'].predict(dte, iteration_range=(0, K['m2'].best_iteration + 1)); SG.append(np.maximum((q[:, 1] - q[:, 0]) / 3.29, 0.05))
    return np.concatenate(MU), (np.concatenate(SG) if SG else None)
def dkm_kur(b, bs=400_000):
    egt, dog, test, gizle = bolunme(b); egt = egt[R_w[egt] > 0]
    if AYAR['hizli_deneme']: egt, test = egt[:200_000], test[:20_000]
    g_gor = torch.zeros(NG + 1, dtype=torch.bool, device=DEV); g_gor[site_genus[ser_site[R_ser[egt]]]] = True
    y_all = hedef(egt); site_e = ser_site[R_ser[egt]]; u = torch.zeros(S, device=DEV)
    for it in range(3):
        XtX = 0; Xty = 0
        for i in range(0, len(egt), bs):
            Z, s = dkm_ozellik(egt[i:i+bs], False, g_gor); rr = y_all[i:i+bs] - u[s]
            XtX = XtX + Z.T.double() @ Z.double(); Xty = Xty + Z.T.double() @ rr.double()
        p_ = XtX.shape[0]; beta = torch.linalg.solve(XtX + 1e-3 * torch.eye(p_, device=DEV, dtype=torch.float64) * XtX.diagonal().mean(), Xty).float()
        res = torch.cat([y_all[i:i+bs] - dkm_ozellik(egt[i:i+bs], False, g_gor)[0] @ beta for i in range(0, len(egt), bs)])
        n_s = torch.bincount(site_e, minlength=S).float(); top = torch.bincount(site_e, res, minlength=S)
        gor = n_s > 0; rbar = top[gor] / n_s[gor]
        s2e = float(((res - (top / n_s.clamp_min(1))[site_e]) ** 2).sum() / (len(res) - gor.sum()))
        s2u = max(float(rbar.var() - (s2e / n_s[gor]).mean()), 1e-4)
        u = torch.zeros(S, device=DEV); u[gor] = n_s[gor] * s2u / (n_s[gor] * s2u + s2e) * rbar
    return dict(beta=beta, u=u, n_s=n_s, g_gor=g_gor, s2e=s2e, s2u=s2u, test=test, gizle=gizle)
@torch.no_grad()
def dkm_tahmin(K, idx, gizle=False, bs=400_000):
    MU = []
    for i in range(0, len(idx), bs):
        Z, s = dkm_ozellik(idx[i:i+bs], gizle, K['g_gor']); MU.append((Z @ K['beta'] + K['u'][s]).cpu().numpy())
    return np.concatenate(MU)

In [ ]:
# --- Ortam bilgisi: kullanılan sürümler (yeniden üretilebilirlik) ---
import sys, platform, importlib
def surum(p):
    try: return importlib.import_module(p).__version__
    except Exception: return 'yok'
SURUM = {p: surum(p) for p in ['numpy', 'pandas', 'scipy', 'torch', 'xgboost', 'xarray', 'zarr', 'fsspec', 'gcsfs']}
SURUM.update(python=platform.python_version(), cuda=(torch.version.cuda if torch.cuda.is_available() else 'yok'))
print(SURUM); json.dump(SURUM, open(os.path.join(ANALIZ, 'H0_surumler.json'), 'w'), indent=1)

In [ ]:
# --- 1. Tüm veriyle (1901–2024) son modellerin eğitimi: U+S, T+S, T+S yerel uyum × 5 tohum; XGBoost (ayarlı) ve karma model ---
for tohum in range(5):
    for m in ['serbest_ortam', 'teori_ortam', 'teori_uyum_ortam']:
        egit('tum', m, tohum)
yol_ayar = os.path.join(ANALIZ, 'G4_xgb_ayar_arama.csv')
if os.path.exists(yol_ayar):
    en = pd.read_csv(yol_ayar).sort_values('dog_rmse').iloc[0]
    PRM = {k: (int(en[k]) if k in ('max_depth', 'max_bin') else float(en[k])) for k in ['max_depth', 'eta', 'subsample', 'colsample_bytree', 'max_bin', 'min_child_weight', 'reg_lambda']}
else:
    PRM = dict(XGB_VARSAYILAN)
kat_gor_ayarla(bolunme('tum')[0])
yol_xgb = os.path.join(SONUC, 'tum__xgb_model.json')
if os.path.exists(yol_xgb):
    KX = dict(m1=xgb.Booster(), m2=None); KX['m1'].load_model(yol_xgb); KX['tur'] = int(json.load(open(yol_xgb + '.tur'))['tur'])
else:
    KX = xgb_kur('tum', PRM, n_egt=6_000_000, n_tur=4000, nicel=False)
    KX['tur'] = int(KX['m1'].best_iteration); KX['m1'].save_model(yol_xgb); json.dump(dict(tur=KX['tur'], prm=PRM), open(yol_xgb + '.tur', 'w'))
yol_dkm = os.path.join(SONUC, 'tum__dkm.pt')
if os.path.exists(yol_dkm): KD = torch.load(yol_dkm, map_location=DEV)
else:
    KD = dkm_kur('tum'); torch.save({k: v for k, v in KD.items() if k not in ('test',)}, yol_dkm)
print('Tüm veriyle modeller hazır. XGBoost tur:', KX['tur'])

In [ ]:
# --- 2. CMIP6 (Google Cloud açık arşivi, CC BY 4.0): 5 iklim modeli × tarihsel + 3 SSP; aylık tas, pr, hurs; alan noktalarında ---
GCM = ['GFDL-ESM4', 'IPSL-CM6A-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'UKESM1-0-LL']
SSP = ['ssp126', 'ssp245', 'ssp585']; DEG3 = ['tas', 'pr', 'hurs']
KLS = os.path.join(ANA, 'CMIP6'); os.makedirs(KLS, exist_ok=True)
LATs, LONs = d['site_lat'].astype(float), d['site_lon'].astype(float)
def cmip_indir():
    import subprocess
    for paket in ['zarr', 'gcsfs', 'cftime']:  # Colab'da zarr ve cftime kurulu gelmiyor
        try: importlib.import_module(paket)
        except ImportError: subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', paket], check=False)
    importlib.invalidate_caches()
    import xarray as xr, fsspec
    try:                                                   # yeni kurulan zarr motorunu xarray'e tanıt (önbellekleri temizle)
        import xarray.namedarray.utils as _nu; _nu.module_available.cache_clear()
    except Exception: pass
    try: xr.backends.plugins.list_engines.cache_clear()
    except Exception: pass
    print('zarr', importlib.import_module('zarr').__version__, '| motorlar:', list(xr.backends.list_engines()))
    KAT = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
    KAT = KAT[(KAT.table_id == 'Amon') & KAT.variable_id.isin(DEG3) & KAT.experiment_id.isin(['historical'] + SSP)]
    lisans = []
    def ac(zs):
        try: return xr.open_zarr(zs, consolidated=True, storage_options={'token': 'anon'}, decode_times=False)
        except Exception as h1_:
            try: return xr.open_zarr(zs.replace('gs://', 'https://storage.googleapis.com/'), consolidated=True, decode_times=False)
            except Exception: raise h1_
    def noktalar(ds, v, y0, y1):
        import cftime                                   # takvimi (noleap, 360_day, …) kendimiz çözüyoruz
        tt = ds['time']; tarih = cftime.num2date(tt.values, tt.attrs['units'], tt.attrs.get('calendar', 'standard'))
        yy = np.array([x.year for x in tarih]); mm = np.array([x.month for x in tarih])
        k_ = np.where((yy >= y0) & (yy <= y1))[0]
        da = ds[v].isel(time=k_).squeeze(drop=True)
        la = 'lat' if 'lat' in da.dims else 'latitude'; lo = 'lon' if 'lon' in da.dims else 'longitude'
        lons = LONs % 360 if float(da[lo].max()) > 180 else LONs
        da = da.sel({la: xr.DataArray(LATs, dims='s'), lo: xr.DataArray(lons, dims='s')}, method='nearest').transpose('time', 's').load()
        out = np.full((y1 - y0 + 1, 12, S), np.nan, np.float32)
        out[yy[k_] - y0, mm[k_] - 1, :] = da.values
        return out
    for g in GCM:
        q = KAT[KAT.source_id == g]
        uyeler = sorted(q.member_id.unique(), key=lambda m_: (m_ != 'r1i1p1f1', m_ != 'r1i1p1f2', m_))
        uye = next((m_ for m_ in uyeler if all(((q.member_id == m_) & (q.experiment_id == e) & (q.variable_id == v)).any()
                                                for e in ['historical'] + SSP for v in DEG3)), None)
        if uye is None: print('ATLANDI (eksik veri):', g); continue
        def zs(e, v): return q[(q.member_id == uye) & (q.experiment_id == e) & (q.variable_id == v)].sort_values('version').iloc[-1].zstore
        f_h = os.path.join(KLS, f'{g}_historical.npz')
        if not os.path.exists(f_h):
            np.savez_compressed(f_h, **{v: noktalar(ac(zs('historical', v)), v, 1985, 2014) for v in DEG3})
        for e in SSP:
            f_e = os.path.join(KLS, f'{g}_{e}.npz')
            if not os.path.exists(f_e):
                np.savez_compressed(f_e, **{v: noktalar(ac(zs(e, v)), v, 2015, 2100) for v in DEG3})
            print('tamam:', g, e, uye)
        lisans.append(dict(model=g, uye=uye, lisans='CC BY 4.0 (CMIP6, Ekim 2022 itibarıyla tüm modelleme grupları)'))
    pd.DataFrame(lisans).to_csv(os.path.join(ANALIZ, 'H0_cmip6_modeller.csv'), index=False)
def cmip_sahte():
    """Yalnız yerel deneme: ağ erişimi yoksa basit yapay değişim (gerçek analizde KULLANILMAZ)"""
    for g in GCM:
        np.savez_compressed(os.path.join(KLS, f'{g}_historical.npz'), tas=np.full((30, 12, S), 285., np.float32), pr=np.full((30, 12, S), 3e-5, np.float32), hurs=np.full((30, 12, S), 70., np.float32))
        for e, dT in zip(SSP, [1.0, 2.0, 4.0]):
            r_ = np.linspace(0, 1, 86)[:, None, None]
            np.savez_compressed(os.path.join(KLS, f'{g}_{e}.npz'), tas=(285. + dT * r_ + np.zeros((86, 12, S))).astype(np.float32),
                                pr=(3e-5 * (1 + 0.05 * r_) + np.zeros((86, 12, S))).astype(np.float32), hurs=(70. - 3 * r_ + np.zeros((86, 12, S))).astype(np.float32))
try:
    cmip_indir()
except Exception as hata:
    if AYAR['hizli_deneme']: print('CMIP6 indirilemedi (yerel deneme) → yapay değişim:', type(hata).__name__); cmip_sahte()
    else: raise
MEVCUT = [g for g in GCM if all(os.path.exists(os.path.join(KLS, f'{g}_{e}.npz')) for e in ['historical'] + SSP)]
print('Kullanılabilir iklim modelleri:', MEVCUT)

In [ ]:
# --- 3. Gelecek iklimi: fark (delta) yöntemi, CRU 1955–2014 serisine; toprak nemi, PET ve VPD yeniden hesaplanır ---
CL = d['clim']; GE = d['gE'].astype(np.float32)          # [S, 124, 12, 4]: T, P, PET, VPD ; [S, 12]
R0 = 1955 - YIL0; NY60 = 60
T_b = CL[:, R0:R0 + NY60, :, 0].astype(np.float64); P_b = CL[:, R0:R0 + NY60, :, 1].astype(np.float64); V_b = CL[:, R0:R0 + NY60, :, 3].astype(np.float64)
I_b = isi_indeksi(T_b.reshape(S, -1))
def kova2(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); PE = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for t_ in range(N):
        Tt, Pt, Lt = T[:, t_], P[:, t_], L[:, t_ % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, t_] = m; PE[:, t_] = Ep
    return M, PE
def iklim(dT=None, rP=None, dRH=None):
    """dT, rP, dRH: [12, S] (aylık). Dönüş: X kanalları [S, 60*12, 5] ve son 30 yılın normalleri"""
    T, P, V = T_b.copy(), P_b.copy(), V_b.copy()
    if dT is not None:
        esT = es(T); RH = np.clip((esT - V) / np.maximum(esT, 1e-6), 0.02, 1.0)
        T = T + dT.T[:, None, :]; P = P * rP.T[:, None, :]; RH = np.clip(RH + dRH.T[:, None, :] / 100, 0.02, 1.0); V = es(T) * (1 - RH)
    M, PE = kova2(T.reshape(S, -1), P.reshape(S, -1), LATs, I_b)
    X = np.stack([T.reshape(S, -1), np.log1p(np.maximum(P.reshape(S, -1), 0)), M, V.reshape(S, -1), np.tile(GE, (1, NY60))], -1).astype(np.float32)
    son = slice(30 * 12, 60 * 12)
    Pa = P.reshape(S, -1)[:, son].reshape(S, 30, 12).sum(2).mean(1); PEa = PE[:, son].reshape(S, 30, 12).sum(2).mean(1)
    nor = dict(T=T.reshape(S, -1)[:, son].mean(1), logP=np.log1p(Pa), logPET=np.log1p(PEa), kur=Pa / np.maximum(PEa, 1), V=V.reshape(S, -1)[:, son].mean(1), M=M[:, son].mean(1))
    return X, nor
def deltalar(g, e, y0, y1):
    h = np.load(os.path.join(KLS, f'{g}_historical.npz')); f = np.load(os.path.join(KLS, f'{g}_{e}.npz'))
    hb = {v: np.nanmean(h[v], 0) for v in DEG3}; fb = {v: np.nanmean(f[v][y0 - 2015:y1 - 2015 + 1], 0) for v in DEG3}
    return fb['tas'] - hb['tas'], np.clip(fb['pr'] / np.maximum(hb['pr'], 1e-12), 0.3, 3.0), fb['hurs'] - hb['hurs']
X_taban, NOR_taban = iklim()
print('taban iklim hazır:', X_taban.shape)

In [ ]:
# --- 4. Ağaç örneklemi: son halkası 1995 ve sonrası olan seriler; alan başına en çok 4 ağaç; ağacın bugünkü yarıçapı ---
R_ser_n = R_ser.cpu().numpy(); R_w_n = R_w.cpu().numpy(); R_rp_n = R_rp.cpu().numpy()
sira = np.lexsort((YIL_n, R_ser_n)); son_i = sira[np.r_[np.flatnonzero(np.diff(R_ser_n[sira])), len(sira) - 1]]
son_i = son_i[YIL_n[son_i] >= 1995]
agac_n = ser_tree.cpu().numpy()[R_ser_n[son_i]]
_, ilk = np.unique(agac_n, return_index=True); son_i = son_i[ilk]
rr = np.random.default_rng(11); son_i = rr.permutation(son_i)
alan_say = {}; sec_i = []
for i_ in son_i:
    s_ = SITE_n[i_]
    if alan_say.get(s_, 0) < 4: sec_i.append(i_); alan_say[s_] = alan_say.get(s_, 0) + 1
N_AGAC = 3000 if AYAR['hizli_deneme'] else 12000
sec_i = np.array(sec_i[:N_AGAC])
A_s = SITE_n[sec_i]; A_ser = R_ser_n[sec_i]; A_r = R_rp_n[sec_i] + R_w_n[sec_i] / 10           # cm
print(f'{len(sec_i)} ağaç, {len(np.unique(A_s))} alan; yarıçap medyanı {np.median(A_r):.1f} cm')
At_s = t(A_s, torch.long); At_ser = t(A_ser, torch.long); At_r = t(A_r, torch.float32)
r99 = pd.Series(R_rp_n + R_w_n / 10).groupby(site_genus.cpu().numpy()[SITE_n]).quantile(0.99)

In [ ]:
# --- 5. Tahmin fonksiyonları (X verilerek): sinir ağları, XGBoost, karma model ---
DESC_ham = desc.astype(np.float64); D_ort, D_ss = DESC_ham.mean(0), DESC_ham.std(0)
def desc_guncel(nor_f):
    """Gelecek normalleri: taban (1985–2014) ile gelecek arasındaki fark, 1901–1960 tanımlayıcılarına eklenir"""
    D = DESC_ham.copy()
    D[:, 2] += nor_f['T'] - NOR_taban['T']; D[:, 3] += nor_f['logP'] - NOR_taban['logP']; D[:, 4] += nor_f['logPET'] - NOR_taban['logPET']
    D[:, 5] *= nor_f['kur'] / np.maximum(NOR_taban['kur'], 1e-6); D[:, 6] += nor_f['V'] - NOR_taban['V']; D[:, 7] += nor_f['M'] - NOR_taban['M']
    return t((D - D_ort) / D_ss, torch.float32), t(MBAR.cpu().numpy() + (nor_f['M'] - NOR_taban['M']), torch.float32), D
def agac_iklimi(Xs):
    """Xs: [S, ay, 5] → ağaçların iklimi [N, ay, 5] (bir kez hesaplanır)"""
    return Xs[At_s]
def pencere_X(XA, j):
    """XA: [N, ay, 5]; j ≥ 1 → yıl j'nin 24 aylık penceresi (önceki yıl Ocak – bu yıl Aralık) [N, 24, 5]"""
    return XA[:, (j - 1) * 12:(j + 1) * 12]
def G_kur(X, DS=None, MB=None):
    G = alan_girdi(At_s)
    if DS is not None: G['desc'] = DS[At_s]
    if MB is not None: G['mbar'] = MB[At_s]
    G.update(X=X, rp=At_r, ser=At_ser); return G
def oz_X(X, DS=None):
    w = X[..., 4]; s = At_s
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    D = DESC[s] if DS is None else DS[s]
    return torch.cat([X[..., :4].flatten(1), torch.log(At_r + 0.1)[:, None]] + mev + [D, ENV[s], site_genus[s].float()[:, None], site_species[s].float()[:, None]], 1), mev, D
@torch.no_grad()
def dkm_X(X, DS=None):
    _, mev, D = oz_X(X, DS); lr = torch.log(At_r + 0.1)[:, None]
    Z = torch.cat([torch.ones_like(lr), lr, lr ** 2] + mev + [D, ENV[At_s], F.one_hot(site_genus[At_s], NG + 1).float()], 1)
    return (Z @ KD['beta'] + KD['u'][At_s]).cpu().numpy()
def xgb_X(X, DS=None):
    Z = oz_X(X, DS)[0].cpu().numpy(); dm = xgb.DMatrix(Z, feature_types=FT, enable_categorical=True)
    return KX['m1'].predict(dm, iteration_range=(0, KX['tur'] + 1))
AGLAR = {}
for m in ['serbest_ortam', 'teori_ortam', 'teori_uyum_ortam']:
    AGLAR[m] = [y_ for y_ in (yukle('tum', m, s_)[0] for s_ in range(5)) if y_ is not None]
    print(m, len(AGLAR[m]), 'tohum')
def tahmin_hepsi(X, DS=None, MB=None):
    """Bir yıl için tüm modellerin log halka genişliği tahmini (sinir ağlarında tohum ortalaması)"""
    out = {}
    G = G_kur(X, DS, MB)
    for m, L in AGLAR.items():
        if L: out[m] = np.mean([tahmin_G(mm_, G).cpu().numpy() for mm_ in L], 0)
    out['xgb'] = xgb_X(X, DS); out['dkm'] = dkm_X(X, DS)
    return out

In [ ]:
# --- 6. ANA SONUÇ (B planı): sabit boyutta iklim etkisi — her ağaç bugünkü yarıçapında, 29 yıllık taban ve gelecek iklimi ---
YILLAR = range(31, 60)
def ortalama_tahmin(Xs, DS=None, MB=None):
    Xs = agac_iklimi(Xs); top = None
    for j in YILLAR:
        o = tahmin_hepsi(pencere_X(Xs, j), DS, MB)
        top = o if top is None else {k: top[k] + o[k] for k in o}
    return {k: v / len(YILLAR) for k, v in top.items()}
Xt_taban = t(X_taban, torch.float32); BAZ = ortalama_tahmin(Xt_taban)
yol_h1 = os.path.join(ANALIZ, 'H1_sabit_boyut_agac.npz'); SONUCLAR = dict(np.load(yol_h1)) if os.path.exists(yol_h1) else {}
NOV = {}
NORM6 = DESC_ham[:, 2:]; mu6 = NORM6.mean(0); Ci6 = np.linalg.inv(np.cov(NORM6, rowvar=False) + 1e-9 * np.eye(6))
dm6 = lambda Z: np.sqrt(np.einsum('ij,jk,ik->i', Z - mu6, Ci6, Z - mu6)); q99 = np.quantile(dm6(NORM6), 0.99)
for g in MEVCUT:
    for e in SSP:
        for (y0, y1) in [(2041, 2070), (2071, 2100)]:
            dT, rP, dRH = deltalar(g, e, y0, y1)
            Xf, nor_f = iklim(dT, rP, dRH); Xt_f = t(Xf, torch.float32); DS, MB, Dh = desc_guncel(nor_f)
            NOV[f'{g}|{e}|{y0}'] = dm6(Dh[:, 2:]) / q99
            for nv, (ds_, mb_) in [('sabit', (None, None)), ('guncel', (DS, MB))]:
                anahtar = f'{g}|{e}|{y0}|{nv}'
                if any(k.startswith(anahtar + '|') for k in SONUCLAR): continue
                F_ = ortalama_tahmin(Xt_f, ds_, mb_)
                for m in F_: SONUCLAR[f'{anahtar}|{m}'] = (F_[m] - BAZ[m]).astype(np.float32)
            np.savez_compressed(yol_h1, **SONUCLAR)
            print('tamam:', g, e, y0)
np.savez_compressed(os.path.join(ANALIZ, 'H1_yenilik_alan.npz'), **{k: v.astype(np.float32) for k, v in NOV.items()})

In [ ]:
# --- 7. Özet tablolar: model × senaryo; iklim modelleri arası; soğuk/kurak çeyrek; yenilik; modeller arası yön anlaşmazlığı ---
soguk = -desc[A_s, 2] >= np.quantile(-desc[A_s, 2], 0.75); kurak = -desc[A_s, 5] >= np.quantile(-desc[A_s, 5], 0.75)
MODEL_AD = {'serbest_ortam': 'U+S', 'teori_ortam': 'T+S', 'teori_uyum_ortam': 'T+S uyumlu', 'xgb': 'XGBoost', 'dkm': 'Karma model'}
h1 = []
for k, v in SONUCLAR.items():
    g, e, y0, nv, m = k.split('|'); nov = NOV.get(f'{g}|{e}|{y0}', np.zeros(S))[A_s] > 1
    yuz = 100 * (np.exp(v) - 1)
    h1.append(dict(gcm=g, ssp=e, donem=f'{y0}–{int(y0) + 29}', normaller=nv, model=MODEL_AD.get(m, m),
                   medyan_degisim_yuzde=float(np.median(yuz)), azalan_oran=float((v < 0).mean()),
                   soguk_ceyrek_medyan=float(np.median(yuz[soguk])), soguk_ceyrek_azalan=float((v[soguk] < 0).mean()),
                   kurak_ceyrek_medyan=float(np.median(yuz[kurak])), kurak_ceyrek_azalan=float((v[kurak] < 0).mean()),
                   yeni_iklim_agac_orani=float(nov.mean()), yeni_iklimde_medyan=float(np.median(yuz[nov])) if nov.any() else np.nan))
h1 = pd.DataFrame(h1); h1.to_csv(os.path.join(ANALIZ, 'H1_sabit_boyut_ozet.csv'), index=False)
h1g = h1.groupby(['ssp', 'donem', 'normaller', 'model']).agg(medyan_degisim_yuzde=('medyan_degisim_yuzde', 'mean'), gcm_arasi_min=('medyan_degisim_yuzde', 'min'),
       gcm_arasi_max=('medyan_degisim_yuzde', 'max'), azalan_oran=('azalan_oran', 'mean'), soguk_ceyrek_azalan=('soguk_ceyrek_azalan', 'mean'),
       kurak_ceyrek_azalan=('kurak_ceyrek_azalan', 'mean'), yeni_iklim_agac_orani=('yeni_iklim_agac_orani', 'mean')).reset_index()
h1g.to_csv(os.path.join(ANALIZ, 'H2_sabit_boyut_gcm_ortalama.csv'), index=False)
print(h1g[(h1g.donem == '2071–2100') & (h1g.normaller == 'sabit')].round(3).to_string(index=False))
# modeller arası yön anlaşmazlığı (alan düzeyi, GCM ortalaması)
h3 = []
for e in SSP:
    for y0 in ['2041', '2071']:
        for nv in ['sabit', 'guncel']:
            ort = {m: np.mean([SONUCLAR[f'{g}|{e}|{y0}|{nv}|{m}'] for g in MEVCUT if f'{g}|{e}|{y0}|{nv}|{m}' in SONUCLAR], 0) for m in MODEL_AD}
            al = pd.DataFrame({m: v for m, v in ort.items()}).groupby(A_s).mean()
            for a_, b_ in [('teori_ortam', 'serbest_ortam'), ('teori_ortam', 'xgb'), ('teori_ortam', 'dkm'), ('serbest_ortam', 'xgb')]:
                if a_ in al and b_ in al:
                    h3.append(dict(ssp=e, donem=y0, normaller=nv, a=MODEL_AD[a_], b=MODEL_AD[b_], zit_yon_alan_orani=float((np.sign(al[a_]) != np.sign(al[b_])).mean()),
                                   spearman=float(spearmanr(al[a_], al[b_])[0])))
h3 = pd.DataFrame(h3); h3.to_csv(os.path.join(ANALIZ, 'H3_model_anlasmazligi.csv'), index=False)
print(h3[(h3.donem == '2071') & (h3.normaller == 'sabit')].round(3).to_string(index=False))
# harita için alan düzeyi değişim (GCM ortalaması, SSP5-8.5, 2071–2100, sabit normaller)
harita = {m: pd.Series(np.mean([SONUCLAR[f'{g}|ssp585|2071|sabit|{m}'] for g in MEVCUT], 0)).groupby(A_s).mean() for m in MODEL_AD}
pd.DataFrame(harita).assign(lat=LATs[np.unique(A_s)], lon=LONs[np.unique(A_s)]).to_csv(os.path.join(ANALIZ, 'H4_harita_ssp585_2071.csv'))

In [ ]:
# --- 8. İKİNCİL SONUÇ: ağaçları 2025–2100 büyütmek (serbest yürütme); SSP2-4.5 ve SSP5-8.5, sabit normaller ---
YIL_G = np.arange(2025, 2101)
def yillik_seri(g=None, e=None):
    """CRU 1955–2014 (ısınma dönemi) + 2025–2100: taban yılı 1985–2014 döngüsü + 30 yıllık kayan GCM değişimi"""
    T, P, V = T_b.copy(), P_b.copy(), V_b.copy()                            # [S, 60, 12]
    by = 30 + (YIL_G - 2025) % 30
    Tf, Pf, Vf = T[:, by].copy(), P[:, by].copy(), V[:, by].copy()            # [S, 76, 12]
    if g is not None:
        h = np.load(os.path.join(KLS, f'{g}_historical.npz')); f = np.load(os.path.join(KLS, f'{g}_{e}.npz'))
        hb = {v: np.nanmean(h[v], 0) for v in DEG3}
        for k_, y in enumerate(YIL_G):
            a_, b_ = max(2015, y - 15), min(2100, y + 14)
            fb = {v: np.nanmean(f[v][a_ - 2015:b_ - 2015 + 1], 0) for v in DEG3}
            dT, rP, dRH = fb['tas'] - hb['tas'], np.clip(fb['pr'] / np.maximum(hb['pr'], 1e-12), 0.3, 3.0), fb['hurs'] - hb['hurs']
            esT = es(Tf[:, k_]); RH = np.clip((esT - Vf[:, k_]) / np.maximum(esT, 1e-6), 0.02, 1)
            Tf[:, k_] += dT.T; Pf[:, k_] *= rP.T; RH = np.clip(RH + dRH.T / 100, 0.02, 1); Vf[:, k_] = es(Tf[:, k_]) * (1 - RH)
    TT = np.concatenate([T, Tf], 1).reshape(S, -1); PP = np.concatenate([P, Pf], 1).reshape(S, -1); VV = np.concatenate([V, Vf], 1).reshape(S, -1)
    M, _ = kova2(TT, PP, LATs, I_b)
    X = np.stack([TT, np.log1p(np.maximum(PP, 0)), M, VV, np.tile(GE, (1, 60 + len(YIL_G)))], -1).astype(np.float32)
    return t(X, torch.float32)
def buyut(Xs):
    """Her model için ağaçları yıl yıl büyüt; dönüş: toplam bazal alan artımı (cm²) ve son yarıçap"""
    global At_r
    r_bas = At_r.clone(); out = {}; Xs = agac_iklimi(Xs)
    for m in ['serbest_ortam', 'teori_ortam', 'xgb', 'dkm']:
        At_r = r_bas.clone(); top = torch.zeros_like(At_r)
        for k_ in range(len(YIL_G)):
            X = pencere_X(Xs, 60 + k_)
            if m in AGLAR: mu = np.mean([tahmin_G(mm_, G_kur(X)).cpu().numpy() for mm_ in AGLAR[m]], 0)
            elif m == 'xgb': mu = xgb_X(X)
            else: mu = dkm_X(X)
            w_cm = t(np.exp(mu) / 10, torch.float32); top += math.pi * ((At_r + w_cm) ** 2 - At_r ** 2); At_r = At_r + w_cm
        out[m] = (top.cpu().numpy(), At_r.cpu().numpy())
    At_r = r_bas
    return out
yol_h5 = os.path.join(ANALIZ, 'H5_serbest_yurutme.npz'); SY = dict(np.load(yol_h5)) if os.path.exists(yol_h5) else {}
if 'taban|teori_ortam|bai' not in SY:
    for m, (bai, rs) in buyut(yillik_seri()).items(): SY[f'taban|{m}|bai'] = bai; SY[f'taban|{m}|r'] = rs
    np.savez_compressed(yol_h5, **SY)
for g in MEVCUT:
    for e in ['ssp245', 'ssp585']:
        if f'{g}|{e}|teori_ortam|bai' in SY: continue
        for m, (bai, rs) in buyut(yillik_seri(g, e)).items(): SY[f'{g}|{e}|{m}|bai'] = bai; SY[f'{g}|{e}|{m}|r'] = rs
        np.savez_compressed(yol_h5, **SY); print('serbest yürütme tamam:', g, e)
h5 = []
gen_A = site_genus.cpu().numpy()[A_s]; esik = r99.reindex(gen_A).values
for g in MEVCUT:
    for e in ['ssp245', 'ssp585']:
        for m in ['serbest_ortam', 'teori_ortam', 'xgb', 'dkm']:
            lr_ = np.log(SY[f'{g}|{e}|{m}|bai'] / np.maximum(SY[f'taban|{m}|bai'], 1e-6))
            h5.append(dict(gcm=g, ssp=e, model=MODEL_AD[m], bai_degisim_medyan_yuzde=float(100 * (np.exp(np.median(lr_)) - 1)), azalan_oran=float((lr_ < 0).mean()),
                           soguk_azalan=float((lr_[soguk] < 0).mean()), kurak_azalan=float((lr_[kurak] < 0).mean()),
                           boyut_disi_oran=float((SY[f'{g}|{e}|{m}|r'] > esik).mean())))
h5 = pd.DataFrame(h5); h5.to_csv(os.path.join(ANALIZ, 'H5_serbest_yurutme_ozet.csv'), index=False)
print(h5.groupby(['ssp', 'model']).mean(numeric_only=True).round(3).to_string())
print('==== DEFTER 8 TAMAM ====  Dosyalar:', ANALIZ)